# Stage 12 — VideoMAE-base + LoRA + landmark fusion (Kaggle T4 / L4 / A100)

Representation-side intervention: VideoMAE-base spatially mean-pooled, temporally upsampled to T=32, concatenated with the existing 190-d landmark stream, projected to 256-d, then the Stage-9a-winning Conformer encoder + joint CTC + attention decoder head.  LoRA on q/k/v (~1.5M trainable / 86M total).

Hyperparameters in the encoder, decoder, λ_ctc, label smoothing are **frozen** from the Stage 9a ablation winners + Stage 11.  The variable here is the backbone, LoRA, fusion, and the two-LR optimizer.

## Prerequisites (attach both as datasets, right panel → Add Data)
1. `wita-handcrop-cache-videomae` — produced by `extract_handcrops_videomae_kaggle.ipynb`.
2. `wita-full-english-landmark-cache` — produced by `extract_landmarks_122_kaggle.ipynb`.

## Test-set discipline
Per §7 of the Stage 12 prompt the test set is touched **exactly once** at the very end of training, using the best-val checkpoint, with no hyperparameter changes in response to test numbers.  Cell 7 is gated by a marker file in `/kaggle/working/logs/.stage12_test_evaluated`.

## Wall-clock
| Phase | T4 | L4 | A100 |
|---|---|---|---|
| Pip + clone + sanity | <3 min | <3 min | <3 min |
| Training: 50 epochs × ~570 steps | ~14 h | ~10 h | ~5 h |
| Final test eval (once) | ~5 min | ~4 min | ~2 min |

## Outputs
- `/kaggle/working/checkpoints/stage12_best.pt`
- `/kaggle/working/logs/stage12_training.json` + `_full.json`
- `/kaggle/working/logs/stage12_test_headline.json`  ← the three CER numbers
- `/kaggle/working/logs/stage12_test_full.json`
- `/kaggle/working/logs/.stage12_test_evaluated` (marker — gates re-runs)

## Cell 1 — Install + clone (VideoMAE + PEFT pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
!pip install 'transformers>=4.40,<4.50' 'peft>=0.11,<0.13' 'accelerate>=0.30' --quiet

import sys, os, glob, json
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import torch, transformers, peft
print(f'GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'transformers : {transformers.__version__}')
print(f'peft         : {peft.__version__}')

## Cell 2 — Locate both caches

In [ ]:
def _find_dir(name):
    cs = (glob.glob(f'/kaggle/working/**/{name}', recursive=True)
          + glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return cs[0] if cs else None

HANDCROP_ROOT = _find_dir('handcrop_cache')
LANDMARK_ROOT = _find_dir('landmark_cache_122')
assert HANDCROP_ROOT, 'handcrop_cache/ not found — attach wita-handcrop-cache-videomae.'
assert LANDMARK_ROOT, 'landmark_cache_122/ not found — attach wita-full-english-landmark-cache.'
print(f'handcrop_root : {HANDCROP_ROOT}')
print(f'landmark_root : {LANDMARK_ROOT}')

# Per-split clip counts so we can see at a glance whether the join lines up.
from pathlib import Path
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        nv = len(list((Path(HANDCROP_ROOT)  / split / subset).glob('*.npz')))
        nl = len(list((Path(LANDMARK_ROOT) / split / subset).glob('*.npz')))
        flag = 'OK' if nv == nl else 'MISMATCH'
        print(f'  {split}/{subset:<7s}  video={nv:>5d}  landmark={nl:>5d}  [{flag}]')

## Cell 3 — Sanity: a single dual-cache batch loads end-to-end

In [ ]:
import logging, random
import numpy as np
from wita_v2.training.stage12_train import WiTAPaperSplitDualDataset, _collate_dual
from wita_v2.datasets.vocab          import make_converter
from wita_v2.configs.default         import Config, DataConfig, EncoderConfig, TrainConfig

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=50, batch_size=16, lr=5e-4,
                      weight_decay=5e-2, grad_clip=1.0,
                      num_workers=2, warmup_pct=0.05, seed=SEED,
                      checkpoint_dir='/kaggle/working/checkpoints'),
).build()
print(f'Device       : {cfg.device}')

converter = make_converter('english')
val_ds = WiTAPaperSplitDualDataset(
    HANDCROP_ROOT, LANDMARK_ROOT, 'val',
    subsets=('lex', 'nonlex'), converter=converter, T_native=32,
)
v, l, ids, signer, subset, label = val_ds[0]
print(f'video    : {tuple(v.shape)}  {v.dtype}    (expected (16,3,224,224) uint8)')
print(f'landmark : {tuple(l.shape)}  {l.dtype}    (expected (32,190) float32)')
print(f'label_ids: {tuple(ids.shape)}  {ids.dtype}  label_str={label!r}')
print(f'signer={signer}  subset={subset}')
from torch.utils.data import DataLoader
coll = lambda b: _collate_dual(b, pad_idx=cfg.vocab.pad_idx)
loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=coll)
batch  = next(iter(loader))
print(f'\nbatch video    : {tuple(batch[0].shape)}  {batch[0].dtype}')
print(f'batch landmark : {tuple(batch[1].shape)}  {batch[1].dtype}')
print(f'batch labels   : {tuple(batch[2].shape)}  {batch[2].dtype}')

## Cell 4 — Sanity: a single forward pass + loss

In [ ]:
from wita_v2.models.stage12_model     import Stage12Model
from wita_v2.models.attention_decoder import build_attention_targets
import torch.nn as nn, torch.nn.functional as F

model = Stage12Model(
    ctc_vocab_size  = cfg.vocab.ctc_vocab_size,
    attn_vocab_size = cfg.vocab.attn_vocab_size,
    sos_idx         = cfg.vocab.sos_idx,
    eos_idx         = cfg.vocab.eos_idx,
    gradient_checkpointing = True,
).to(cfg.device)
print(f'Total trainable params: {model.num_trainable:,}')

videos, landmarks, labels, in_lens, lab_lens, _, _, _ = batch
videos    = videos.to(cfg.device)
landmarks = landmarks.to(cfg.device)
labels    = labels.to(cfg.device); in_lens = in_lens.to(cfg.device); lab_lens = lab_lens.to(cfg.device)
dec_in, dec_tg = build_attention_targets(
    labels, lab_lens, bos=cfg.vocab.sos_idx, eos=cfg.vocab.eos_idx, pad=cfg.vocab.pad_idx,
)
with torch.cuda.amp.autocast(dtype=torch.float16, enabled=torch.cuda.is_available()):
    log_probs, enc_lens, dec_logits, _, _ = model(videos, landmarks, in_lens, dec_in)
    ctc = nn.CTCLoss(blank=cfg.vocab.blank_idx, zero_infinity=True)
    ce  = nn.CrossEntropyLoss(ignore_index=cfg.vocab.pad_idx, label_smoothing=0.1)
    ctc_l  = ctc(log_probs.transpose(0,1).float(), labels, enc_lens, lab_lens)
    attn_l = ce(dec_logits.reshape(-1, model.decoder.att_vocab_size), dec_tg.reshape(-1))
print(f'\nlog_probs : {tuple(log_probs.shape)}  enc_lens={enc_lens.tolist()}')
print(f'dec_logits: {tuple(dec_logits.shape)}')
print(f'ctc_loss  : {float(ctc_l):.4f}')
print(f'attn_loss : {float(attn_l):.4f}')
del model, videos, landmarks, labels, in_lens, lab_lens, dec_in, dec_tg
del log_probs, enc_lens, dec_logits, ctc_l, attn_l
torch.cuda.empty_cache()

## Cell 5 — Config

Hyperparameters frozen from `configs/stage12.yaml`.  Do not edit by hand without bumping the file + the §7 contract.

In [ ]:
LOG_DIR  = '/kaggle/working/logs'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage12.log'))])

VARIANT_NAME = 'stage12'
NUM_EPOCHS   = 50
BATCH_SIZE   = 16
LAMBDA_CTC   = 0.5     # Stage 9a winner
DEC_N_LAYERS = 2       # Stage 9a winner
DEC_N_HEADS  = 4
LR_BACKBONE  = 5e-5    # LoRA + post-LN
LR_HEAD      = 5e-4    # fusion + encoder + decoder
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
WARMUP_PCT   = 0.05
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
UPSAMPLE     = 2
DROPOUT      = 0.2
T_NATIVE     = 32
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.1
AMP_DTYPE    = 'fp16'  # 'bf16' on A100
GRAD_CKPT    = True
VIDEOMAE_NAME = 'MCG-NJU/videomae-base'

print(f'Variant      : {VARIANT_NAME}')
print(f'lambda_ctc   : {LAMBDA_CTC}')
print(f'lr           : backbone {LR_BACKBONE:.0e}  head {LR_HEAD:.0e}')
print(f'epochs/batch : {NUM_EPOCHS} / {BATCH_SIZE}')
print(f'amp_dtype    : {AMP_DTYPE}')

## Cell 6 — Train  (val checkpoint selection on val_overall_cer)

Idempotent skip: if `/kaggle/working/logs/stage12_training.json` exists (committed prior version) the cell loads its `checkpoint_path` and `best_payload` instead of re-running.

In [ ]:
TRAINING_JSON = os.path.join(LOG_DIR, f'{VARIANT_NAME}_training.json')

def _reuse_or_run_training():
    if os.path.exists(TRAINING_JSON):
        with open(TRAINING_JSON) as f:
            return json.load(f)
    # Also accept a prior committed version surfaced via /kaggle/input/**.
    cands = glob.glob(f'/kaggle/input/**/{VARIANT_NAME}_training.json', recursive=True)
    if cands:
        import shutil
        shutil.copy(cands[0], TRAINING_JSON)
        # also pull the checkpoint if it's alongside
        cdir = os.path.dirname(cands[0])
        for sub in ('checkpoints', '.'):
            cks = glob.glob(os.path.join(cdir, '..', sub, f'{VARIANT_NAME}_best.pt'))
            if cks:
                shutil.copy(cks[0], os.path.join(CKPT_DIR, f'{VARIANT_NAME}_best.pt'))
                break
        with open(TRAINING_JSON) as f:
            return json.load(f)
    return None

result = _reuse_or_run_training()
if result is not None:
    print(f'[reuse] training already complete (best_val_overall='
          f'{result.get("best_val_overall", "?"):.4f}, epoch '
          f'{result.get("best_epoch", "?")})')
else:
    from wita_v2.training.stage12_train import train_stage12
    result = train_stage12(
        handcrop_root=HANDCROP_ROOT,
        landmark_root=LANDMARK_ROOT,
        cfg=cfg,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
        warmup_pct=WARMUP_PCT, lambda_ctc=LAMBDA_CTC, label_smoothing=0.1,
        videomae_model_name=VIDEOMAE_NAME,
        lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, dropout=DROPOUT,
        dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
        T_native=T_NATIVE,
        amp_dtype=AMP_DTYPE, gradient_checkpointing=GRAD_CKPT,
        num_workers=2, seed=SEED,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR, variant=VARIANT_NAME,
    )
    print('\n=== Best on validation ===')
    print(json.dumps(result['best_payload'], indent=2, default=str))

## Cell 7 — Test evaluation  (EXACTLY ONCE — DO NOT RE-RUN)

Per §7 of the Stage 12 prompt, this cell must run exactly once.  A marker file at `/kaggle/working/logs/.stage12_test_evaluated` is created when it succeeds; re-running will raise a `RuntimeError` unless the marker is deleted by hand.

**Stop and write the three CER numbers down before doing anything else.**

In [ ]:
HEADLINE_PATH = os.path.join(LOG_DIR, f'{VARIANT_NAME}_test_headline.json')

def _reuse_or_run_test():
    if os.path.exists(HEADLINE_PATH):
        with open(HEADLINE_PATH) as f:
            return {'headline': json.load(f), '_reused': True}
    cands = glob.glob(f'/kaggle/input/**/{VARIANT_NAME}_test_headline.json', recursive=True)
    if cands:
        import shutil; shutil.copy(cands[0], HEADLINE_PATH)
        full = cands[0].replace('_test_headline.json', '_test_full.json')
        if os.path.exists(full):
            shutil.copy(full, os.path.join(LOG_DIR, f'{VARIANT_NAME}_test_full.json'))
        with open(HEADLINE_PATH) as f:
            return {'headline': json.load(f), '_reused': True}
    return None

reused = _reuse_or_run_test()
if reused is not None:
    print(f'[reuse] test headline already on disk ({HEADLINE_PATH})')
    test_out = reused
    # Hydrate per-signer / per-length for the diagnostics cell from the full file.
    full_path = os.path.join(LOG_DIR, f'{VARIANT_NAME}_test_full.json')
    if os.path.exists(full_path):
        with open(full_path) as f:
            full = json.load(f)
        test_out['per_signer_cer'] = full.get('per_signer_cer', {})
        test_out['per_length_cer'] = full.get('per_length_cer', {})
else:
    from wita_v2.training.stage12_train import final_test_eval
    ckpt = result['checkpoint_path']
    test_out = final_test_eval(
        handcrop_root=HANDCROP_ROOT, landmark_root=LANDMARK_ROOT,
        checkpoint=ckpt, cfg=cfg,
        batch_size=BATCH_SIZE, T_native=T_NATIVE, amp_dtype=AMP_DTYPE,
        num_workers=2,
        log_dir=LOG_DIR, variant=VARIANT_NAME,
        videomae_model_name=VIDEOMAE_NAME,
        lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, dropout=DROPOUT, upsample=UPSAMPLE,
        dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
    )

## Cell 8 — Band classification (only after Cell 7 wrote the headline)

In [ ]:
h = test_out['headline']
ov, lex, non = h['test_overall_cer'], h['test_lex_cer'], h['test_nonlex_cer']
paper = h['paper_baseline']
s11   = h.get('stage11_baseline', {'overall': 0.4498, 'lex': 0.4348})
print(f'\nyour overall = {ov:.4f}   (paper {paper["overall"]}, S11 {s11["overall"]})')
print(f'your lex     = {lex:.4f}   (paper {paper["lex"]},     S11 {s11["lex"]})')
print(f'your nonlex  = {non:.4f}   (paper {paper["nonlex"]})')
if ov <= 0.32 and lex <= 0.30:
    band, nxt = 'STRONG_PASS', 'write the thesis section; optionally chain Stage 9b KenLM rescoring'
elif ov <= 0.38 and lex <= 0.35:
    band, nxt = 'PASS', 'add Stage 9b KenLM for a small overall lift; consider unfreezing more VideoMAE blocks'
elif ov <= 0.42:
    band, nxt = 'PARTIAL', 'increase LoRA r to 32 OR unfreeze last 2 blocks OR augment the video stream'
else:
    band, nxt = 'UNDERPERFORMS', 'diagnose: per-signer scatter, per-length CER vs S11, video-only vs landmark-only ablation'
print(f'\nBand: {band}')
print(f'Recommended next step: {nxt}')
delta_paper = paper['overall'] - ov
delta_s11   = s11['overall']   - ov
print(f'\nΔ vs paper : {delta_paper:+.4f}  (positive = better than paper)')
print(f'Δ vs S11   : {delta_s11:+.4f}  (positive = beats landmark-only baseline)')

## Cell 9 — Diagnostics figures (read but do NOT use to override Cell 7)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Length-bucketed test CER vs Stage 11 (typed in for reference).
bucket_cer = test_out.get('per_length_cer', {})
bucket_order = ['1-4','5-8','9-12','13-inf']
bs = [b for b in bucket_order if b in bucket_cer]
ys = [bucket_cer[b] for b in bs]
if bs:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(bs, ys, color='#1f77b4')
    for x, y in zip(bs, ys): ax.text(x, y+0.005, f'{y:.3f}', ha='center', fontsize=8)
    ax.set_xlabel('label length bucket'); ax.set_ylabel('test CER')
    ax.set_title('Stage 12 — length-bucketed test CER')
    ax.grid(True, linestyle=':', alpha=0.4, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, 'stage12_length_buckets.png'), dpi=140)
    plt.show()

# Per-signer test CER.
per_signer = test_out.get('per_signer_cer', {})
items = sorted(per_signer.items(), key=lambda kv: kv[1])
if items:
    fig, ax = plt.subplots(figsize=(14, 4.0))
    ax.scatter(range(len(items)), [v for _, v in items], s=18, color='#2ca02c')
    ax.axhline(0.40, color='green', linestyle='--', alpha=0.4, label='strong ≤ 0.40')
    ax.axhline(0.60, color='red',   linestyle='--', alpha=0.4, label='tail ≥ 0.60')
    ax.set_xlabel('signer (sorted by CER)'); ax.set_ylabel('test CER')
    ax.set_title(f'Stage 12 — per-signer test CER  (n={len(items)} test signers)')
    ax.set_xticks([]); ax.grid(True, linestyle=':', alpha=0.3)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, 'stage12_per_signer_test.png'), dpi=140)
    plt.show()

## Cell 10 — Commit kernel

Save Version → Save & Run All to preserve:
- `logs/stage12_test_headline.json`  ← the three CER numbers
- `logs/stage12_test_full.json`
- `logs/stage12_training.json` + `_full.json`
- `logs/stage12_length_buckets.png`, `stage12_per_signer_test.png`
- `checkpoints/stage12_best.pt`
- `logs/.stage12_test_evaluated`  ← marker, prevents accidental re-eval

Send `stage12_test_headline.json` here for the writeup.